# 🖼️ Image Enhancement & Upscaling

This notebook enhances low-quality images through a **5-stage pipeline**:

| Stage | Technique | Purpose |
|---|---|---|
| 1 | Load & Inspect | Preview input image and metadata |
| 2 | Denoising | Non-Local Means in L\*a\*b space |
| 3 | Sharpening | Unsharp masking for edge crispness |
| 4 | Contrast & Color | CLAHE + saturation boost |
| 5 | Upscaling | Lanczos + bilateral filter refinement |

---

## 📦 Install & Import Dependencies

In [ ]:
# Install required packages (run once)
import subprocess
subprocess.run(['pip', 'install', 'opencv-python-headless', 'Pillow', 'numpy', 'matplotlib', '-q'])
print('✅ Dependencies ready')

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

print('✅ All imports successful')
print(f'   OpenCV  : {cv2.__version__}')
print(f'   Pillow  : {Image.__version__}')
print(f'   NumPy   : {np.__version__}')

---
## ⚙️ Configuration

Set your input image path and tweak the parameters below.

In [ ]:
# ──────────────────────────────────────────────
#  INPUT / OUTPUT  (resolved relative to this notebook's folder)
# ──────────────────────────────────────────────
from pathlib import Path
import os

# Folder where this notebook lives
NOTEBOOK_DIR = Path(globals().get('__vsc_ipynb_file__',
                    os.environ.get('JPY_SESSION_NAME', __file__ if '__file__' in dir() else '.'))).parent.resolve()
# Fallback: use current working directory (works in JupyterLab / classic Notebook)
if not NOTEBOOK_DIR.exists():
    NOTEBOOK_DIR = Path.cwd()

INPUT_FILENAME  = 'your_image.jpg'   # ← only the filename, no folder needed
INPUT_PATH      = str(NOTEBOOK_DIR / INPUT_FILENAME)
OUTPUT_PATH     = None               # None = auto-generate in the same folder

# ──────────────────────────────────────────────
#  PIPELINE PARAMETERS
# ──────────────────────────────────────────────
SCALE           = 8.0   # Upscale factor: 2 = 2×, 4 = 4×
DENOISE_STR     = 8     # Denoise strength: 0 = off, 1-5 mild, 6-12 strong
SHARPEN_AMOUNT  = 1.5   # Sharpening: 1.0 = none, 1.5 = mild, 2.5 = strong
CLAHE_CLIP      = 2.5   # Contrast enhancement clip limit (1.0–4.0)
SATURATION      = 1.25  # Color saturation multiplier (1.0 = no change)

# ──────────────────────────────────────────────
#  TOGGLE STAGES ON / OFF
# ──────────────────────────────────────────────
DO_DENOISE = True
DO_SHARPEN = True
DO_COLOR   = False

print('✅ Configuration set')

---
## 🔧 Pipeline Functions

In [ ]:
def bgr_to_rgb(img: np.ndarray) -> np.ndarray:
    """Convert OpenCV BGR to matplotlib-friendly RGB."""
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def show_comparison(images: list, titles: list, figsize=(18, 5)):
    """Display multiple images side-by-side with titles and size info."""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        h, w = img.shape[:2]
        ax.imshow(bgr_to_rgb(img) if img.ndim == 3 else img, cmap='gray')
        ax.set_title(f'{title}\n{w}×{h} px', fontsize=11, fontweight='bold')
        ax.axis('off')
    plt.tight_layout()
    plt.show()


def stage_denoise(img: np.ndarray, strength: float = 8) -> np.ndarray:
    """
    Non-Local Means denoising in L*a*b colour space.
    Luminance denoised aggressively; chroma gently to avoid colour smearing.
    """
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    L, a, b = cv2.split(lab)
    L = cv2.fastNlMeansDenoising(L, h=strength,           templateWindowSize=7, searchWindowSize=21)
    a = cv2.fastNlMeansDenoising(a, h=strength * 0.4,     templateWindowSize=7, searchWindowSize=21)
    b = cv2.fastNlMeansDenoising(b, h=strength * 0.4,     templateWindowSize=7, searchWindowSize=21)
    return cv2.cvtColor(cv2.merge([L, a, b]), cv2.COLOR_LAB2BGR)


def stage_sharpen(img: np.ndarray, amount: float = 1.5) -> np.ndarray:
    """
    Unsharp masking: subtract a blurred version to enhance edges.
    Blur radius scales with image size.
    """
    r = max(1, int(min(img.shape[:2]) * 0.006)) | 1  # must be odd
    blurred = cv2.GaussianBlur(img, (r, r), 0)
    sharpened = cv2.addWeighted(img, amount, blurred, -(amount - 1), 0)
    return np.clip(sharpened, 0, 255).astype(np.uint8)


def stage_contrast_color(img: np.ndarray, clahe_clip: float = 2.5, sat: float = 1.25) -> np.ndarray:
    """
    CLAHE on luminance for local contrast, then HSV saturation boost.
    """
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    L, a, b = cv2.split(lab)
    tile = max(4, min(img.shape[0], img.shape[1]) // 64)
    clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=(tile, tile))
    L = clahe.apply(L)
    enhanced = cv2.cvtColor(cv2.merge([L, a, b]), cv2.COLOR_LAB2BGR)

    hsv = cv2.cvtColor(enhanced, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:, :, 1] = np.clip(hsv[:, :, 1] * sat, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)


def stage_upscale(img: np.ndarray, scale: float = 2.0) -> np.ndarray:
    """
    Lanczos upscaling (Pillow) + bilateral filter to reduce ringing.
    """
    if scale <= 1.0:
        return img
    h, w = img.shape[:2]
    new_w, new_h = int(round(w * scale)), int(round(h * scale))
    pil = Image.fromarray(bgr_to_rgb(img))
    pil_up = pil.resize((new_w, new_h), Image.LANCZOS)
    up = cv2.cvtColor(np.array(pil_up), cv2.COLOR_RGB2BGR)
    d = max(5, int(min(new_h, new_w) * 0.005)) | 1
    return cv2.bilateralFilter(up, d=d, sigmaColor=40, sigmaSpace=40)


print('✅ Pipeline functions defined')

---
## 📂 Stage 1 — Load & Inspect Input Image

In [ ]:
img_original = cv2.imread(INPUT_PATH, cv2.IMREAD_COLOR)
if img_original is None:
    raise FileNotFoundError(f'Cannot read: {INPUT_PATH}  — check the path!')

h0, w0 = img_original.shape[:2]
size_kb = Path(INPUT_PATH).stat().st_size / 1024

print(f'📄 File     : {INPUT_PATH}')
print(f'📐 Size     : {w0} × {h0} px')
print(f'💾 Disk     : {size_kb:.1f} KB')
print(f'🎨 Channels : {img_original.shape[2] if img_original.ndim==3 else 1}')

img_current = img_original.copy()
show_comparison([img_original], ['Original (Input)'])

---
## 🔇 Stage 2 — Noise Reduction

In [ ]:
if DO_DENOISE:
    print(f'⏳ Denoising (strength={DENOISE_STR}) — may take a few seconds ...')
    t = time.time()
    img_denoised = stage_denoise(img_current, strength=DENOISE_STR)
    print(f'✅ Done in {time.time()-t:.2f}s')
    show_comparison([img_current, img_denoised], ['Before Denoise', 'After Denoise'])
    img_current = img_denoised
else:
    print('⏭️  Denoising skipped (DO_DENOISE = False)')

---
## ✏️ Stage 3 — Sharpening

In [ ]:
if DO_SHARPEN:
    print(f'⏳ Sharpening (amount={SHARPEN_AMOUNT}) ...')
    t = time.time()
    img_sharp = stage_sharpen(img_current, amount=SHARPEN_AMOUNT)
    print(f'✅ Done in {time.time()-t:.2f}s')
    show_comparison([img_current, img_sharp], ['Before Sharpening', 'After Sharpening'])
    img_current = img_sharp
else:
    print('⏭️  Sharpening skipped (DO_SHARPEN = False)')

---
## 🎨 Stage 4 — Contrast & Color Enhancement

In [ ]:
if DO_COLOR:
    print(f'⏳ Enhancing contrast (CLAHE clip={CLAHE_CLIP}) and saturation (×{SATURATION}) ...')
    t = time.time()
    img_color = stage_contrast_color(img_current, clahe_clip=CLAHE_CLIP, sat=SATURATION)
    print(f'✅ Done in {time.time()-t:.2f}s')
    show_comparison([img_current, img_color], ['Before Color Enhance', 'After Color Enhance'])
    img_current = img_color
else:
    print('⏭️  Color enhancement skipped (DO_COLOR = False)')

---
## 🔍 Stage 5 — Upscaling

In [ ]:
print(f'⏳ Upscaling ×{SCALE} ({w0}×{h0} → {int(w0*SCALE)}×{int(h0*SCALE)}) ...')
t = time.time()
img_upscaled = stage_upscale(img_current, scale=SCALE)
h1, w1 = img_upscaled.shape[:2]
print(f'✅ Done in {time.time()-t:.2f}s  |  Output: {w1}×{h1} px')

show_comparison(
    [img_original, img_upscaled],
    [f'Original  ({w0}×{h0})', f'Enhanced & Upscaled  ({w1}×{h1})'],
    figsize=(14, 6)
)

img_final = img_upscaled

---
## 📊 Histogram Analysis — Before vs After

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
colors = ('b', 'g', 'r')
labels = ('Blue', 'Green', 'Red')

for img, ax, title in zip(
        [img_original, img_final],
        axes,
        ['Original', 'Enhanced']):
    for c, col, lbl in zip(range(3), colors, labels):
        hist = cv2.calcHist([img], [c], None, [256], [0, 256])
        ax.plot(hist, color=col, alpha=0.7, label=lbl)
    ax.set_title(title, fontweight='bold')
    ax.set_xlim([0, 256])
    ax.set_xlabel('Pixel Intensity')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('RGB Histogram Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 💾 Save Output

In [ ]:
if OUTPUT_PATH is None:
    p = Path(INPUT_PATH)
    OUTPUT_PATH = str(NOTEBOOK_DIR / f'{p.stem}_enhanced_{SCALE:.0f}x{p.suffix}')

ext = Path(OUTPUT_PATH).suffix.lower()
params = []
if ext in ('.jpg', '.jpeg'):
    params = [cv2.IMWRITE_JPEG_QUALITY, 95]
elif ext == '.png':
    params = [cv2.IMWRITE_PNG_COMPRESSION, 6]
elif ext == '.webp':
    params = [cv2.IMWRITE_WEBP_QUALITY, 95]

ok = cv2.imwrite(OUTPUT_PATH, img_final, params)
if ok:
    out_kb = Path(OUTPUT_PATH).stat().st_size / 1024
    print(f'✅ Saved → {OUTPUT_PATH}')
    print(f'   Size  : {w1}×{h1} px | {out_kb:.0f} KB')
else:
    print(f'❌ Failed to save to {OUTPUT_PATH}')

---
## 📋 Summary Report

In [ ]:
in_kb  = Path(INPUT_PATH).stat().st_size / 1024
out_kb = Path(OUTPUT_PATH).stat().st_size / 1024 if Path(OUTPUT_PATH).exists() else 0

print('=' * 48)
print('         ENHANCEMENT SUMMARY')
print('=' * 48)
print(f'  Input   : {INPUT_PATH}')
print(f'  Output  : {OUTPUT_PATH}')
print(f'  Resolution : {w0}×{h0}  →  {w1}×{h1}  (×{SCALE})')
print(f'  File size  : {in_kb:.0f} KB  →  {out_kb:.0f} KB')
print('─' * 48)
print(f'  Denoise : {"ON  (strength=" + str(DENOISE_STR) + ")" if DO_DENOISE else "OFF"}')
print(f'  Sharpen : {"ON  (amount=" + str(SHARPEN_AMOUNT) + ")" if DO_SHARPEN else "OFF"}')
print(f'  Color   : {"ON  (CLAHE=" + str(CLAHE_CLIP) + ", sat=" + str(SATURATION) + ")" if DO_COLOR else "OFF"}')
print(f'  Upscale : ×{SCALE}  (Lanczos + bilateral)')
print('=' * 48)